In [0]:
class Bronze_qualifying():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze" 
    
    def __init__(self,folder_name,source):
        self.folder_name=folder_name
        self.source=source
    
    def get_schema(self):
        from pyspark.sql.types import IntegerType, StringType, DateType, StructType, StructField
        qualifying_schema = StructType([
            StructField("qualifyId", IntegerType(), False),        # Primary Key, NOT NULL
            StructField("raceId", IntegerType(), False),           # Foreign Key, NOT NULL
            StructField("driverId", IntegerType(), False),         # Foreign Key, NOT NULL
            StructField("constructorId", IntegerType(), False),    # Foreign Key, NOT NULL
            StructField("number", IntegerType(), False),           # NOT NULL
            StructField("position", IntegerType(), True),          # Nullable
            StructField("q1", StringType(), True),                 # Nullable
            StructField("q2", StringType(), True),                 # Nullable
            StructField("q3", StringType(), True)                  # Nullable
        ])
        return qualifying_schema

    def read_data(self):
        df= (spark.readStream
             .format("json")
             .schema(self.get_schema())
             .option('multiline','true')
             .option("maxFilesPerTrigger", 1)
             #.option("rowsPerSecond", 1000)
             .load(f"{self.main_path}/formula1_source/{self.folder_name}")
             )
        return df
        
    def process(self):
        print(f"\nStarting Bronze qualifying Stream...", end='')
        readDF = self.read_data()
        from pyspark.sql.functions import current_timestamp,lit
        readDF= (readDF.withColumn('QualifyingIngestionDate',current_timestamp())
                 .withColumn('source',lit(self.source))
                 )
        sQuery =  ( readDF.writeStream
                            .queryName("bronze-ingestion-qualifying")
                            .option("checkpointLocation", f"{self.main_path}/{self.bronze_path}/{self.folder_name}/checkpoint")
                            .outputMode("append")
                            #.option('path',f"{self.main_path}/{self.bronze_path}/{self.folder_name}")
                            .trigger(availableNow=True)
                            .toTable('formula1_race.bronze.qualifying')
                                
                    ) 
        print("Done")
        return sQuery   

In [0]:
from pyspark.sql.streaming import StreamingQueryListener

class MyListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"Query started: {event.id}")

    def onQueryProgress(self, event):
        print(f"Batch ID: {event.progress['batchId']}")
        print(f"Rows Read: {event.progress['numInputRows']}")
        print(f"Duration (ms): {event.progress['durationMs']}")
        print(f"Rows/sec: {event.progress['processedRowsPerSecond']}")

    def onQueryTerminated(self, event):
        print(f"Query terminated: {event.id}")

spark.streams.addListener(MyListener())

In [0]:
source='Ergast API'
Bronze_qualifying_instance = Bronze_qualifying("qualifying",source)
Squery_Bronze_qualifying =Bronze_qualifying_instance.process()
Squery_Bronze_qualifying.awaitTermination()
print("Successfully bronze-ingestion-qualifying stream in running")
Squery_Bronze_qualifying.stop()